# Simulación Multifísica de Ondas Acústicas Superficiales (SAW) Tipo Rayleigh en Heteroestructuras AlN / Si(111)

Este notebook implementa:
1. **Definición de tensores constitutivos** para AlN (wurtzita $6mm$, eje $c$ orientado en $Z$) y Silicio cúbico ($m\bar{3}m$).
2. **Transformación de Bond ($6 \times 6$)** para rotar el tensor de rigidez del Silicio al plano cristalográfico **Si(111)**.
3. **Aproximación semi-analítica** del perfil de desplazamiento superficial.
4. **Solucionador numérico 1D SAFE / FDM acoplado** con endurecimiento piezoeléctrico para determinar frecuencia de resonancia $f_0$, velocidad de fase $v_p$, coeficiente de acoplamiento electromecánico $K^2$ y curvas de dispersión $h_{\text{AlN}}/\lambda$.

In [ ]:
import numpy as np

def generar_tensores_aln():
    """
    Define los tensores para Nitruro de Aluminio (AlN) - Simetría 6mm (Wurtzita).
    El eje c (0002) está orientado en el eje Z.
    """
    # Matriz de rigidez elástica C^E (Pa)
    C_AlN = np.array([
        [345, 125, 120,   0,   0,   0],
        [125, 345, 120,   0,   0,   0],
        [120, 120, 395,   0,   0,   0],
        [  0,   0,   0, 118,   0,   0],
        [  0,   0,   0,   0, 118,   0],
        [  0,   0,   0,   0,   0, 110]  # C66 = (C11 - C12)/2
    ], dtype=float) * 1e9

    # Matriz piezoeléctrica e (C/m^2)
    e_AlN = np.array([
        [    0,     0,    0,     0, -0.48, 0],
        [    0,     0,    0, -0.48,     0, 0],
        [-0.58, -0.58, 1.55,     0,     0, 0]
    ], dtype=float)

    # Permitividad dieléctrica relativa y absoluta (F/m)
    eps_rel_AlN = np.array([
        [9.0,   0,    0],
        [  0, 9.0,    0],
        [  0,   0, 10.7]
    ], dtype=float)
    eps_0 = 8.8541878128e-12
    eps_AlN = eps_rel_AlN * eps_0

    rho_AlN = 3260.0  # kg/m^3

    return C_AlN, e_AlN, eps_AlN, rho_AlN

def construir_matriz_bond(R):
    """
    Construye la matriz de transformación de Bond M (6x6) a partir de la matriz
    de cosenos directores R (3x3) en notación de Voigt:
    1->(0,0), 2->(1,1), 3->(2,2), 4->(1,2), 5->(0,2), 6->(0,1)
    """
    M = np.zeros((6, 6), dtype=float)
    voigt = [(0, 0), (1, 1), (2, 2), (1, 2), (0, 2), (0, 1)]
    for I, (i, j) in enumerate(voigt):
        for J, (k, l) in enumerate(voigt):
            if I < 3 and J < 3:
                M[I, J] = R[i, k] * R[j, l]
            elif I < 3 and J >= 3:
                M[I, J] = 2.0 * R[i, k] * R[j, l]
            elif I >= 3 and J < 3:
                M[I, J] = R[i, k] * R[j, l]
            else:
                M[I, J] = R[i, k] * R[j, l] + R[i, l] * R[j, k]
    return M

def generar_matriz_rotacion_si111():
    """
    Calcula la matriz de rotación espacial hacia el plano (111) del Silicio
    y aplica la transformación de Bond: C_Si_111 = M @ C_Si_cubico @ M^T.
    """
    # Constantes elásticas del Silicio (Cúbico m3m) en Pa
    C11, C12, C44 = 165.7e9, 63.9e9, 79.6e9

    C_Si_cubico = np.zeros((6, 6), dtype=float)
    C_Si_cubico[0:3, 0:3] = C12
    np.fill_diagonal(C_Si_cubico[0:3, 0:3], C11)
    np.fill_diagonal(C_Si_cubico[3:6, 3:6], C44)

    # Ángulos de Euler para alinear Z con [111]
    phi = np.pi / 4.0                      # Rotación de 45° en Z
    theta = np.arccos(1.0 / np.sqrt(3.0))  # Rotación ~54.74° en Y'

    R_z = np.array([
        [ np.cos(phi), np.sin(phi), 0],
        [-np.sin(phi), np.cos(phi), 0],
        [           0,           0, 1]
    ], dtype=float)

    R_y = np.array([
        [np.cos(theta), 0, -np.sin(theta)],
        [            0, 1,              0],
        [np.sin(theta), 0,  np.cos(theta)]
    ], dtype=float)

    matriz_rotacion = R_y @ R_z
    M_bond = construir_matriz_bond(matriz_rotacion)
    C_Si_111 = M_bond @ C_Si_cubico @ M_bond.T
    rho_Si = 2329.0  # kg/m^3

    return C_Si_cubico, C_Si_111, matriz_rotacion, M_bond, rho_Si

# Extraer parámetros constitutivos rotados
C_AlN, e_AlN, eps_AlN, rho_AlN = generar_tensores_aln()
C_Si_cubico, C_Si_111, R_mat, M_bond, rho_Si = generar_matriz_rotacion_si111()

print("Tensores de AlN(0002) y matriz rotada C_Si(111) listos.")
print(f"C11_Si(111) = {C_Si_111[0,0]/1e9:.2f} GPa | C33_Si(111) = {C_Si_111[2,2]/1e9:.2f} GPa | C44_Si(111) = {C_Si_111[3,3]/1e9:.2f} GPa")

In [ ]:
import matplotlib.pyplot as plt

# 1. Parámetros del dominio computacional (aproximación semi-analítica)
lambda_saw = 4.0                     # Longitud de onda de diseño en micrómetros
k = 2 * np.pi / lambda_saw           # Número de onda
h_AlN = 0.5                          # Espesor de la película delgada de AlN en micrómetros

x = np.linspace(0, lambda_saw, 200)
z = np.linspace(-lambda_saw * 1.5, h_AlN + 0.2, 300)
X, Z = np.meshgrid(x, z)

# 2. Modelado de los campos de desplazamiento
q = 0.8 * k
s = 0.5 * k

# Sustrato de Si(111) (Z < 0)
U_Si_z = np.exp(q * Z) - 0.6 * np.exp(s * Z)
U_Si_x = np.exp(q * Z) - 0.4 * np.exp(s * Z)

# Película delgada de AlN (0 <= Z <= h_AlN)
U_AlN_z = np.cos(k * Z * 0.3) + 0.1
U_AlN_x = np.sin(k * Z * 0.3) + 0.1

# 3. Ensamblaje de la estructura multicapa
Uz = np.where(Z < 0, U_Si_z, U_AlN_z)
Ux = np.where(Z < 0, U_Si_x, U_AlN_x)

Uz = np.where(Z > h_AlN, 0, Uz)
Ux = np.where(Z > h_AlN, 0, Ux)

# 4. Propagación armónica en t=0
U_x_2D = Ux * np.cos(k * X)
U_z_2D = Uz * np.sin(k * X)
U_total = np.sqrt(U_x_2D**2 + U_z_2D**2)
U_total = U_total / np.max(U_total)

# 5. Visualización
plt.figure(figsize=(10, 5.5))
cp = plt.contourf(X, Z, U_total, levels=100, cmap='jet')
cbar = plt.colorbar(cp)
cbar.set_label(r'Magnitud de Desplazamiento Normalizado $|U|$', fontsize=11)

plt.axhline(0, color='white', linestyle='--', linewidth=2.0, label='Interfaz AlN / Si(111)')
plt.axhline(h_AlN, color='black', linestyle='-', linewidth=2.0, label='Superficie Libre (Aire)')

plt.title('Modo Acústico de Rayleigh (Modelo Semi-Analítico): Confinamiento en AlN', fontsize=13, fontweight='bold')
plt.xlabel(r'Distancia Longitudinal $X$ ($\mu m$)', fontsize=11)
plt.ylabel(r'Profundidad $Z$ ($\mu m$)', fontsize=11)
plt.legend(loc='lower right', framealpha=0.9)
plt.ylim(-lambda_saw * 1.5, h_AlN + 0.2)
plt.xlim(0, lambda_saw)
plt.tight_layout()
plt.show()

In [ ]:
from scipy.linalg import eig

def resolver_modo_rayleigh_fdm(lambda_saw, h_AlN, N=180, metalizado=False):
    """
    Solucionador 1D SAFE / Diferencias Finitas Centrales para ondas de Rayleigh
    en la heteroestructura AlN(0002) / Si(111) con endurecimiento piezoeléctrico.
    """
    k = 2.0 * np.pi / lambda_saw
    h_Si = 8.0 * lambda_saw  # Profundidad suficiente para decaimiento completo (> 5 lambda)
    z = np.linspace(-h_Si, h_AlN, N)
    dz = z[1] - z[0]
    is_aln = z >= 0

    # Constantes efectivas de AlN: endurecimiento piezoeléctrico C_eff = C^E + e^T * eps^-1 * e
    if not metalizado:
        C11_AlN_use = C_AlN[0, 0] + (e_AlN[2, 0]**2) / eps_AlN[2, 2]
        C33_AlN_use = C_AlN[2, 2] + (e_AlN[2, 2]**2) / eps_AlN[2, 2]
        C44_AlN_use = C_AlN[3, 3] + (e_AlN[0, 4]**2) / eps_AlN[0, 0]
        C13_AlN_use = C_AlN[0, 2] + (e_AlN[2, 0] * e_AlN[2, 2]) / eps_AlN[2, 2]
    else:
        # Condición de cortocircuito eléctrico (superficie metalizada)
        C11_AlN_use = C_AlN[0, 0]
        C33_AlN_use = C_AlN[2, 2]
        C44_AlN_use = C_AlN[3, 3]
        C13_AlN_use = C_AlN[0, 2]

    # Perfiles espaciales usando el tensor rotado C_Si_111
    rho = np.where(is_aln, rho_AlN, rho_Si)
    C11 = np.where(is_aln, C11_AlN_use, C_Si_111[0, 0])
    C33 = np.where(is_aln, C33_AlN_use, C_Si_111[2, 2])
    C44 = np.where(is_aln, C44_AlN_use, C_Si_111[3, 3])
    C13 = np.where(is_aln, C13_AlN_use, C_Si_111[0, 2])

    L = np.zeros((2 * N, 2 * N), dtype=float)
    M = np.diag(np.repeat(rho, 2))

    # Nodos internos (Diferencias Finitas Centrales)
    for i in range(1, N - 1):
        # Ecuación de movimiento en X (u_x)
        L[2*i, 2*i]     = k**2 * C11[i] + 2.0 * C44[i] / dz**2
        L[2*i, 2*i-2]   = -C44[i] / dz**2
        L[2*i, 2*i+2]   = -C44[i] / dz**2
        L[2*i, 2*i+3]   = k * (C13[i] + C44[i]) / (2.0 * dz)
        L[2*i, 2*i-1]   = -k * (C13[i] + C44[i]) / (2.0 * dz)

        # Ecuación de movimiento en Z (u_z)
        L[2*i+1, 2*i+1] = k**2 * C44[i] + 2.0 * C33[i] / dz**2
        L[2*i+1, 2*i-1] = -C33[i] / dz**2
        L[2*i+1, 2*i+3] = -C33[i] / dz**2
        L[2*i+1, 2*i+2] = -k * (C13[i] + C44[i]) / (2.0 * dz)
        L[2*i+1, 2*i-2] = k * (C13[i] + C44[i]) / (2.0 * dz)

    # Frontera inferior (Base de Si anclada / Dirichlet)
    L[0, 0] = 1e20; M[0, 0] = 1.0
    L[1, 1] = 1e20; M[1, 1] = 1.0

    # Frontera superior (Superficie libre de esfuerzos en z = h_AlN)
    i = N - 1
    L[2*i, 2*i]     = k**2 * C11[i] + 2.0 * C44[i] / dz**2 - k**2 * (C13[i] + C44[i]) * (C13[i] / C33[i])
    L[2*i, 2*i-2]   = -2.0 * C44[i] / dz**2
    L[2*i, 2*i+1]   = -2.0 * C44[i] * k / dz

    L[2*i+1, 2*i+1] = k**2 * C44[i] + 2.0 * C33[i] / dz**2 - k**2 * (C13[i] + C44[i])
    L[2*i+1, 2*i-1] = -2.0 * C33[i] / dz**2
    L[2*i+1, 2*i]   = 2.0 * k * C13[i] / dz

    # Problema generalizado de autovalores
    A = np.linalg.inv(M) @ L
    eigenvalues, eigenvectors = eig(A)

    valid = (np.real(eigenvalues) > 0) & (np.abs(np.imag(eigenvalues)) < 1e-8)
    omegas = np.sqrt(np.real(eigenvalues[valid]))
    vecs = np.real(eigenvectors[:, valid])

    frecuencias_hz = omegas / (2.0 * np.pi)
    velocidades_fase = frecuencias_hz * lambda_saw

    # Filtrar modo fundamental de Rayleigh (menor velocidad de fase física > 4200 m/s)
    mask_rayleigh = (velocidades_fase > 4200.0) & (velocidades_fase < 5800.0)
    f_cand = frecuencias_hz[mask_rayleigh]
    v_cand = velocidades_fase[mask_rayleigh]
    vecs_cand = vecs[:, mask_rayleigh]

    idx_min = np.argmin(v_cand)
    return f_cand[idx_min], v_cand[idx_min], z, vecs_cand[:, idx_min]

# Parámetros geométricos de diseño
lambda_saw = 9.0e-6   # 9 um
h_AlN = 1.0e-6        # 1 um

f_open, v_open, z_mesh, vec_rayleigh = resolver_modo_rayleigh_fdm(lambda_saw, h_AlN, N=180, metalizado=False)
f_short, v_short, _, _ = resolver_modo_rayleigh_fdm(lambda_saw, h_AlN, N=180, metalizado=True)
K2_pct = 2.0 * (v_open - v_short) / v_open * 100.0

print(f"Modo Rayleigh AlN/Si(111) [h/lambda = {h_AlN/lambda_saw:.3f}]:")
print(f"  -> Frecuencia de resonancia (f0): {f_open/1e6:.2f} MHz")
print(f"  -> Velocidad de fase libre (v_open): {v_open:.2f} m/s")
print(f"  -> Velocidad metalizada (v_short):   {v_short:.2f} m/s")
print(f"  -> Acoplamiento electromecánico K^2: {K2_pct:.3f} %")

# Reconstrucción 2D enfocada en la zona activa (-2*lambda <= z <= h_AlN)
u_x_prof = vec_rayleigh[0::2]
u_z_prof = vec_rayleigh[1::2]

mask_zoom = z_mesh >= -2.0 * lambda_saw
z_zoom = z_mesh[mask_zoom]
x_grid = np.linspace(0, lambda_saw, 200)
X_2D, Z_2D = np.meshgrid(x_grid, z_zoom)

U_x_2D = np.outer(u_x_prof[mask_zoom], np.cos(2 * np.pi / lambda_saw * x_grid))
U_z_2D = np.outer(u_z_prof[mask_zoom], np.sin(2 * np.pi / lambda_saw * x_grid))
U_total = np.sqrt(U_x_2D**2 + U_z_2D**2)
U_total /= np.max(U_total)

plt.figure(figsize=(10, 5.5))
cp = plt.contourf(X_2D * 1e6, Z_2D * 1e6, U_total, levels=100, cmap='jet')
plt.colorbar(cp, label=r'Desplazamiento Normalizado $|U|$')
plt.axhline(0, color='white', linestyle='--', linewidth=2.0, label='Interfaz AlN / Si(111)')
plt.title(f'Modo de Rayleigh Acoplado AlN / Si(111) (FDM)\n$f_0$ = {f_open/1e6:.2f} MHz | $v_p$ = {v_open:.1f} m/s | $K^2$ = {K2_pct:.3f}%', fontweight='bold')
plt.xlabel(r'Eje de Propagación $X$ ($\mu m$)')
plt.ylabel(r'Profundidad $Z$ ($\mu m$)')
plt.legend(loc='lower right', framealpha=0.9)
plt.tight_layout()
plt.show()

In [ ]:
# 4. Curva de dispersión: Velocidad de fase y K^2 vs espesor normalizado (h_AlN / lambda)
ratios_h_lambda = np.linspace(0.02, 0.45, 16)
v_open_list = []
k2_list = []

for ratio in ratios_h_lambda:
    h_curr = ratio * lambda_saw
    _, vo, _, _ = resolver_modo_rayleigh_fdm(lambda_saw, h_curr, N=160, metalizado=False)
    _, vs, _, _ = resolver_modo_rayleigh_fdm(lambda_saw, h_curr, N=160, metalizado=True)
    v_open_list.append(vo)
    k2_list.append(2.0 * (vo - vs) / vo * 100.0)

fig, ax1 = plt.subplots(figsize=(9.5, 5))

color1 = 'tab:blue'
ax1.set_xlabel(r'Espesor Normalizado $h_{\mathrm{AlN}} / \lambda$', fontsize=11)
ax1.set_ylabel(r'Velocidad de Fase $v_p$ (m/s)', color=color1, fontsize=11)
ax1.plot(ratios_h_lambda, v_open_list, 'o-', color=color1, linewidth=2, label=r'Velocidad de fase ($v_{\mathrm{open}}$)')
ax1.tick_params(axis='y', labelcolor=color1)
ax1.grid(True, linestyle=':', alpha=0.6)

ax2 = ax1.twinx()
color2 = 'tab:red'
ax2.set_ylabel(r'Acoplamiento Electromecánico $K^2$ (%)', color=color2, fontsize=11)
ax2.plot(ratios_h_lambda, k2_list, 's--', color=color2, linewidth=2, label=r'$K^2$ (%)')
ax2.tick_params(axis='y', labelcolor=color2)

plt.title('Dispersión de Modo Rayleigh y Eficiencia Piezoeléctrica en AlN / Si(111)', fontsize=12, fontweight='bold')
fig.tight_layout()
plt.show()